# Chart Reading Comparison

Runs the structural chart-description prompt over **every chart folder** in this project and collects the LLM's raw output per chart type (`Version_1` ... `Version_4`).

- Works with **any number of images per folder** — every subfolder that contains at least one image is picked up automatically. To test a new chart type, just drop a new folder with images next to this notebook and re-run.
- No scoring or validation: each image's output is printed as it arrives and everything is saved to `llm_outputs_by_version.json` for manual review.


In [ ]:
from openai import OpenAI

# ------------------------------
# CONFIG
# ------------------------------
API_KEY = ""  # <-- paste your key here
BASE_URL = "https://finderllm2-resource.services.ai.azure.com/openai/v1"
MODEL = "gpt-5-mini"
ROOT_FOLDER = "."
OUTPUT_FILE = "llm_outputs_by_version.json"

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)


In [ ]:
# ------------------------------
# PROMPT - dual Y-axis aware
# ------------------------------
PROMPT = """
You are a vision system analyzing a financial chart.

Describe ONLY what is visually present.

For each subplot:
- number of subplots in the full image
- number of time series
- color and label of each series
- which y-axis each series belongs to: "left" or "right"
- x-axis label and visible tick values
- left y-axis label and approximate min/max
- right y-axis label and approximate min/max (only if a second y-axis is visually present)

A dual y-axis exists when:
  - there are two different y-axis scales, one on the left and one on the right side of the plot
  - the right axis has its own label or tick values distinct from the left

Return ONLY valid JSON in this exact format:

{
  "num_plots": int,
  "plots": [
    {
      "plot_id": int,
      "num_series": int,
      "dual_y_axis": bool,
      "series": [
        {
          "color": "...",
          "label": "...",
          "y_axis_side": "left" or "right"
        }
      ],
      "x_axis": {
        "label": "...",
        "values": []
      },
      "y_axis_left": {
        "label": "...",
        "min": float,
        "max": float
      },
      "y_axis_right": {
        "label": "...",
        "min": float,
        "max": float
      }
    }
  ]
}

If there is no right y-axis for a subplot, set:
  "dual_y_axis": false,
  "y_axis_right": {"label": null, "min": null, "max": null}
"""


In [ ]:
import os
import base64

# ------------------------------
# HELPERS
# ------------------------------
VALID_EXT = (".png", ".jpg", ".jpeg", ".webp")

def encode_image(path):
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode()

def guess_mime_type(path):
    p = path.lower()
    if p.endswith(".png"):
        return "image/png"
    if p.endswith(".jpg") or p.endswith(".jpeg"):
        return "image/jpeg"
    if p.endswith(".webp"):
        return "image/webp"
    return "application/octet-stream"

def get_images(folder):
    return sorted(
        os.path.join(folder, f)
        for f in os.listdir(folder)
        if f.lower().endswith(VALID_EXT)
    )

def get_chart_folders(root):
    """Every subfolder of root that contains at least one image (dot-folders skipped)."""
    return [
        os.path.join(root, d)
        for d in sorted(os.listdir(root))
        if not d.startswith(".")
        and os.path.isdir(os.path.join(root, d))
        and get_images(os.path.join(root, d))
    ]

def strip_fences(raw):
    cleaned = raw.strip()
    if cleaned.startswith("```"):
        parts = cleaned.split("```")
        if len(parts) >= 2:
            cleaned = parts[1]
            if cleaned.startswith("json"):
                cleaned = cleaned[4:]
    return cleaned.strip()

chart_folders = get_chart_folders(ROOT_FOLDER)
print(f"Found {len(chart_folders)} chart folder(s):")
for folder in chart_folders:
    print(f"  {os.path.basename(folder)}: {len(get_images(folder))} image(s)")


In [ ]:
import json

# ------------------------------
# PROCESS ALL CHART FOLDERS
# ------------------------------
results = {}

for chart_folder in chart_folders:
    chart_name = os.path.basename(chart_folder)
    images = get_images(chart_folder)
    print(f"\n{'=' * 70}")
    print(f"\U0001F4C1 {chart_name} - {len(images)} image(s)")
    print("=" * 70)

    results[chart_name] = []

    for image_path in images:
        fname = os.path.basename(image_path)
        print(f"\n\U0001F4CA {fname}")

        image_base64 = encode_image(image_path)
        mime = guess_mime_type(image_path)
        response = client.chat.completions.create(
            model=MODEL,
            messages=[{
                "role": "user",
                "content": [
                    {"type": "text", "text": PROMPT},
                    {"type": "image_url", "image_url": {"url": f"data:{mime};base64,{image_base64}"}},
                ],
            }],
        )

        raw_output = response.choices[0].message.content
        cleaned = strip_fences(raw_output)

        try:
            output = json.loads(cleaned)
            print(json.dumps(output, indent=2))
        except json.JSONDecodeError:
            output = raw_output  # keep the raw text so nothing is lost
            print(raw_output)

        results[chart_name].append({"image": fname, "output": output})

with open(OUTPUT_FILE, "w") as f:
    json.dump(results, f, indent=2)

print(f"\n\u2705 All outputs saved to {OUTPUT_FILE}")
